In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../src/battery_cycle_data.csv")

In [2]:
df.head()

,battery_id,cycle,capacity,voltage_mean,voltage_min,voltage_max,voltage_std,current_mean,current_min,current_max,...,temperature_mean,temperature_min,temperature_max,temperature_std,current_load_mean,voltage_load_mean,duration,voltage_slope,ic_peak_height,ic_peak_voltage
0,B0005,1,1.856487,3.529829,2.612467,4.191492,0.236558,-1.818702,-2.018015,0.000729,...,32.572328,24.325993,38.982181,3.495804,-1.805570,2.404944,3690.234,-0.000206,6.351944,3.479329
1,B0005,2,1.846327,3.537320,2.587209,4.189773,0.235366,-1.817560,-2.016821,0.002927,...,32.725235,24.685948,39.033398,3.435509,-1.804583,2.399260,3672.344,-0.000205,6.781573,3.500066
2,B0005,3,1.835349,3.543737,2.651917,4.188187,0.228111,-1.816487,-2.016574,0.001484,...,32.642862,24.734266,38.818797,3.388174,-1.803575,2.397969,3651.641,-0.000200,6.854167,3.482607
3,B0005,4,1.835263,3.543666,2.592948,4.188461,0.233347,-1.825589,-2.015936,0.001547,...,32.514876,24.652244,38.762305,3.395306,-1.812863,2.408289,3631.563,-0.000205,6.988207,3.496433
4,B0005,5,1.834646,3.542343,2.547420,4.188299,0.237301,-1.826114,-2.017426,0.001701,...,32.382349,24.518700,38.665393,3.404667,-1.812876,2.408505,3629.172,-0.000208,6.098681,3.510383


In [3]:
df['SOH'] = (100 * df['capacity']) / df.groupby('battery_id')['capacity'].transform('first')
df.head()

,battery_id,cycle,capacity,voltage_mean,voltage_min,voltage_max,voltage_std,current_mean,current_min,current_max,...,temperature_min,temperature_max,temperature_std,current_load_mean,voltage_load_mean,duration,voltage_slope,ic_peak_height,ic_peak_voltage,SOH
0,B0005,1,1.856487,3.529829,2.612467,4.191492,0.236558,-1.818702,-2.018015,0.000729,...,24.325993,38.982181,3.495804,-1.805570,2.404944,3690.234,-0.000206,6.351944,3.479329,100.000000
1,B0005,2,1.846327,3.537320,2.587209,4.189773,0.235366,-1.817560,-2.016821,0.002927,...,24.685948,39.033398,3.435509,-1.804583,2.399260,3672.344,-0.000205,6.781573,3.500066,99.452721
2,B0005,3,1.835349,3.543737,2.651917,4.188187,0.228111,-1.816487,-2.016574,0.001484,...,24.734266,38.818797,3.388174,-1.803575,2.397969,3651.641,-0.000200,6.854167,3.482607,98.861386
3,B0005,4,1.835263,3.543666,2.592948,4.188461,0.233347,-1.825589,-2.015936,0.001547,...,24.652244,38.762305,3.395306,-1.812863,2.408289,3631.563,-0.000205,6.988207,3.496433,98.856718
4,B0005,5,1.834646,3.542343,2.547420,4.188299,0.237301,-1.826114,-2.017426,0.001701,...,24.518700,38.665393,3.404667,-1.812876,2.408505,3629.172,-0.000208,6.098681,3.510383,98.823482


In [4]:
""" The NASA battery dataset uses 70% (0.70) SOH as its criteria to find the RUL (Remaining Useful Life)
    as per the README.txt for all 4 cells (B0005, B0006, B0007 and B0018).

    Not every battery actually crosses the 70% SOH line within its recorded cycles -
    this is documented behavior for this dataset (one of the four cells never fades
    past ~72% capacity retention before testing stopped). For that battery, EOL_cycle
    and RUL are not "missing data" to be filled in, they're fundamentally unknown -
    this is right-censored data in survival-analysis terms. We keep it as NaN and flag
    it explicitly with a `censored` column, rather than silently leaving NaNs to be
    discovered later (or worse, accidentally imputed/dropped-by-row without noticing
    it's an entire battery).
"""

eol_cycles = df[df['SOH'] <= 70].groupby('battery_id')['cycle'].min()

df['EOL_cycle'] = df['battery_id'].map(eol_cycles)
df['RUL'] = df['EOL_cycle'] - df['cycle']

censored_batteries = sorted(df.loc[df['EOL_cycle'].isna(), 'battery_id'].unique().tolist())
df['censored'] = df['battery_id'].isin(censored_batteries)

print(f"Batteries that never reached 70% SOH in the recorded data (censored): {censored_batteries}")
print(f"Rows with unknown RUL: {df['RUL'].isna().sum()} / {len(df)}")

Batteries that never reached 70% SOH in the recorded data (censored): ['B0007', 'B0018']
Rows with unknown RUL: 300 / 636


In [5]:
df['cycle_normalized'] = (
    df['cycle'] /
    df.groupby('battery_id')['cycle'].transform('max')
)

df['voltage_range'] = (
    df['voltage_max'] - df['voltage_min']
)

df['temperature_range'] = (
    df['temperature_max'] - df['temperature_min']
)

In [6]:
df.isna().sum()

battery_id             0
cycle                  0
capacity               0
voltage_mean           0
voltage_min            0
voltage_max            0
voltage_std            0
current_mean           0
current_min            0
current_max            0
current_std            0
temperature_mean       0
temperature_min        0
temperature_max        0
temperature_std        0
current_load_mean      0
voltage_load_mean      0
duration               0
voltage_slope          0
ic_peak_height         0
ic_peak_voltage        0
SOH                    0
EOL_cycle            300
RUL                  300
censored               0
cycle_normalized       0
voltage_range          0
temperature_range      0
dtype: int64

In [7]:
df.to_csv("../src/final_data.csv", index=False)